In [ ]:
# config
samples = ["A1", "A2", "B2", "C2", "D1"]
input_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/raw_data"
zarr_file_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/20260126_final.zarr"
plotting = True
on_hpc = True
unit_testing = False
gpu = False

In [ ]:
import spatialdata as sd
import harpy as hp
import scanpy as sc

import torch.nn
import torch
import scvi
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection

## Loading preprocessed SpatialData object from Zarr

In [ ]:
sdata = sd.read_zarr(
    zarr_file_path,
    on_bad_files = "warn"
)
sdata

## Filtering Cells & Genes (per sample)

In [ ]:
# for sample in samples:
#     key = f"{sample}_transcriptomics"
#     adata_filtered = sdata.tables[key].copy()
#     sc.pp.filter_cells(adata_filtered, min_counts = 10) # min nr of counts in cells
#     sc.pp.filter_genes(adata_filtered, min_cells = 5) # min nr of genes in cells

#     hp.tb.add_table_layer(
#         sdata,
#         adata_filtered,
#         output_layer = f"{key}_filtered",
#         overwrite = True,
#         region = None
#     )
# sdata

In [ ]:
for sample in samples:
    sdata = hp.tb.preprocess_transcriptomics(
        sdata = sdata,
        labels_layer = f"{sample}_segmentation_mask",
        table_layer = f"{sample}_transcriptomics",
        output_layer = f"{sample}_transcriptomics_filtered", 
        min_counts = 10,
        min_cells = 5,
        size_norm = True,
        highly_variable_genes = False,
        max_value_scale = None,
        overwrite = True,
        n_comps = None,
        update_shapes_layers = True,
    )

In [ ]:
sdata

## Batch correction using scVI

In [ ]:
# creating a clean merged master adata object
adatas = []
for sample in samples:
    adatas.append(
        sdata.tables[f"{sample}_transcriptomics_filtered"].copy()
    )

for adata in adatas: 
    keep = ["cell_ID", "fov_labels", "n_counts", "shapeSize"]
    adata.obs = adata.obs.loc[:, keep].copy()

adata_master = sc.concat(
    adatas,
    keys = samples,
    label = "sample_id",
    uns_merge = "unique",
    index_unique = None,
)

adata_master.uns.pop("log1p", None)
adata_master.uns.pop("pca", None)

adata_master

In [ ]:
# setting the X to raw counts
adata_master.X = adata_master.layers["raw_counts"].copy()
adata_master.X[:5, :5].toarray()

In [ ]:
# setting up the model
scvi.model.SCVI.setup_anndata(
    adata_scvi, 
    layer = "raw_counts",
    batch_key = "sample_id",
    continuous_covariate_keys = ["shapeSize"]
)

In [ ]:
model_scvi = scvi.model.SCVI(adata_scvi)
model_scvi

In [ ]:
model_scvi.view_anndata_setup()

In [ ]:
if gpu == True:
    import os
    os.environ.pop('SLURM_NTASKS', None) # env variable set (otherwise error if training model on hpc)
    os.environ.pop('SLURM_NTASKS_PER_NODE', None) # env variable set (otherwise error if training model on hpc)
    torch.set_float32_matmul_precision("high") # env variable set (otherwise error if training model on hpc)
    device_accelerator = "gpu"
else:
    device_accelerator = "cpu"

In [ ]:
model_scvi.train(
    accelerator = device_accelerator,
    devices = 1,
    batch_size = 1024,
    max_epochs = 400,
    check_val_every_n_epoch = 1,
    early_stopping_monitor = "elbo_validation",
    early_stopping_patience = 30
)

In [ ]:
train_test_results = model_scvi.history["elbo_train"]
train_test_results["elbo_validation"] = model_scvi.history["elbo_validation"]
train_test_results.iloc[100:].plot(logy=True) 
plt.show()

In [ ]:
adata_scvi.obsm["X_scVI"] = model_scvi.get_latent_representation()

In [ ]:
adata_scvi

In [ ]:
sc.pp.neighbors(
    adata_scvi, 
    n_neighbors = 40, 
    knn = True, 
    use_rep = "X_scVI"
)
sc.tl.umap(
    adata_scvi
)

sc.tl.leiden(
    adata_scvi, 
    key_added = "clusters", 
    n_iterations = -1, 
    flavor = "igraph", 
    directed = False, 
    resolution = 0.5
)

In [ ]:
umap_integrated = sc.pl.umap(
    adata_scvi, 
    color = [
        "clusters", 
        "sample_id"
    ], 
    return_fig = True,
    show = False,
    # visuals
    palette = sc.pl.palettes.default_20,
    size = 4,
    frameon = False,
    add_outline = False,
    wspace = 0.3,
    hspace = 4
)

In [ ]:
umap_integrated.set_size_inches(8, 6)
umap_integrated.tight_layout()
umap_integrated.savefig("umap_integrated.svg", bbox_inches="tight")

In [ ]:
# saving the model with the Anndata
model_scvi.save("intermediate_results/model_20260128_final", overwrite = True, save_anndata = True)
adata_scvi

## Mapping clusters back to samples

In [ ]:
cluster_key = "clusters"

adatas_with_clusters = {}
for sample in samples:
    # copy the original table
    ad = sdata.tables[f"{sample}_transcriptomics_filtered"].copy()

    # transfer clusters by aligning on obs_names
    ad.obs[cluster_key] = adata_scvi.obs.loc[ad.obs_names, cluster_key].astype("category")

    adatas_with_clusters[sample] = ad

adatas_with_clusters

In [ ]:
# copying UMAP coordinates from adata_scvi to per sample AnnData 
name_to_idx = pd.Series(np.arange(adata_scvi.n_obs), index=adata_scvi.obs_names)

for s, ad in adatas_with_clusters.items():
    idx = name_to_idx.loc[ad.obs_names].values
    ad.obsm["X_umap_scvi"] = adata_scvi.obsm["X_umap"][idx, :]

In [ ]:
cluster_key = "clusters"

for sample in samples:
    # make a copy of the table
    ad = sdata.tables[f"{sample}_transcriptomics_filtered"].copy()

    # add the clusters
    ad.obs[cluster_key] = adata_scvi.obs.loc[ad.obs_names, cluster_key].astype("category")

    # overwrite or save as a new table
    sdata.tables[f"{sample}_transcriptomics_filter_scvi_clusters"] = ad

sdata

## Cell type annotation

In [ ]:
for sample in samples:
    hp.pl.plot_shapes(
        sdata,
        img_layer = f"{sample}_clahe",
        table_layer = f"{sample}_transcriptomics_filter_scvi_clusters",
        column = "clusters",
        shapes_layer=f"{sample}_segmentation_mask_boundaries",
        alpha = 1.0,
        linewidth = 0,
        cmap = 'tab20',
        to_coordinate_system = sample    
    )

In [ ]:
sc.tl.rank_genes_groups(
    adata_scvi,
    groupby = "clusters",          
    method = "wilcoxon",           
    pts = True,                    
    use_raw = True                
)
sc.pl.rank_genes_groups(adata_scvi, n_genes=10, sharey=False)

In [ ]:
cluster_to_celltype = {
    "0": "Oligodendrocytes",
    "1": "CA1 Neurons",
    "2": "Microglia",
    "3": "CA2/CA3 Neurons",
    "4": "Inhibitory Neurons",
    "5": "Cortical Neurons",
    "6": "OPCs",
    "7": "Choroid Plexus Cells",
    "8": "Astrocytes Fibrous",
    "9": "Thalamic Neurons",
    "10": "Astrocytes Protoplasmic",
    "11": "DG Granular Neurons",
    "12": "Medial Habenula Neurons",
    "13": "Endothelial Cells",
    "14": "DG Subgranular Layer"
}

In [ ]:
adata_scvi.obs["cell_type"] = (
    adata_scvi.obs["clusters"].astype(str).map(cluster_to_celltype)
)
adata_scvi.obs["cell_type"] = adata_scvi.obs["cell_type"].astype("category")

In [ ]:
sc.pl.umap(adata_scvi, color="cell_type")

In [ ]:
adata_scvi

## Mapping cell types back to samples and assign colors

In [ ]:
label_cols = ["cell_type", "clusters"] 

for sample in samples:
    table_name = f"{sample}_transcriptomics_filtered"
    out_name = f"{sample}_transcriptomics_filter_scvi_annotated"

    ad = sdata.tables[table_name].copy()

    # transfer labels
    for col in label_cols:
        ad.obs[col] = adata_scvi.obs.loc[ad.obs_names, col].astype("category")

    # store it in the sdata
    sdata.tables[out_name] = ad

In [ ]:
sdata

In [ ]:
# assign colors to each cell type across samples
key = "cell_type"
celltypes = adata_scvi.obs[key].astype("category").cat.categories.tolist()
palette = [plt.get_cmap("tab20")(i) for i in range(len(celltypes))]
tohex = lambda c: "#{:02x}{:02x}{:02x}".format(int(c[0]*255), int(c[1]*255), int(c[2]*255))
colors = [tohex(c) for c in palette]
ct2color = dict(zip(celltypes, colors))

adata_scvi.uns[f"{key}_colors"] = colors

for s in samples:
    ad = sdata.tables[f"{s}_transcriptomics_filter_scvi_annotated"]
    ad.obs[key] = ad.obs[key].astype("category").cat.set_categories(celltypes)
    ad.uns[f"{key}_colors"] = [ct2color[ct] for ct in celltypes]

In [ ]:
for sample in samples:
    hp.pl.plot_shapes(
        sdata,
        img_layer=f"{sample}_clahe",
        table_layer=f"{sample}_transcriptomics_filter_scvi_annotated",
        column="cell_type",
        shapes_layer=f"{sample}_segmentation_mask_boundaries",
        alpha=1.0,
        linewidth=0,
        legend = False,
        to_coordinate_system = sample    
    )

## Saving the model and object

In [ ]:
sdata

In [ ]:
adata_scvi

In [ ]:
# save the model again, now with cell types
model_scvi.save("intermediate_results/model_20260128_final", overwrite = True, save_anndata = True)

In [ ]:
# writing to zarr
sdata.write(f"/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/20260128_final.zarr", overwrite=True)